# Turkish (tur) — Full NLP Pipeline

Turkish is the most resource-rich Turkic language in TurkicNLP, supporting the full processing pipeline: tokenization, multi-word token expansion, morphological analysis (Apertium FST, Production quality), POS tagging, lemmatization, dependency parsing (7 UD treebank variants), named entity recognition, sentence embeddings, and machine translation via NLLB-200.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
# Download all Turkish models (Stanza + Apertium FST)
turkicnlp.download("tur")

## 2. Tokenisation and Multi-Word Token Expansion

In [ ]:
# Rule-based tokeniser (default) with MWT expansion
nlp_tok = Pipeline("tur", processors=["tokenize", "mwt"])

doc = nlp_tok("İstanbul'a gitmek istiyorum.")
for sent in doc.sentences:
    print("Tokens:", [tok.text for tok in sent.tokens])
    print("Words: ", [w.text for w in sent.words])

## 3. Morphological Analysis (Apertium FST)

In [ ]:
turkicnlp.download("tur", processors=["tokenize", "morph"])

nlp_morph = Pipeline(
    "tur",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
)

doc = nlp_morph("Türkiye'nin başkenti Ankara'dır.")
for word in doc.words:
    print(f"{word.text:<20} lemma={word.lemma:<15} feats={word.feats}")

## 4. POS Tagging, Lemmatisation, and Dependency Parsing

Seven UD treebank variants are available for Turkish. The default is `IMST` (general domain). Others: `BOUN`, `FrameNet`, `KeNet`, `ATIS` (aviation), `Penn`, `Tourism`.

In [ ]:
# Default IMST treebank
nlp_parse = Pipeline("tur", processors=["tokenize", "pos", "lemma", "depparse"])

doc = nlp_parse("Ahmet bugün İstanbul'a gitti.")
print(f"{'Word':<15} {'UPOS':<8} {'Lemma':<15} {'Head':<5} {'Deprel'}")
print("-" * 55)
for w in doc.words:
    print(f"{w.text:<15} {w.upos:<8} {w.lemma:<15} {w.head!s:<5} {w.deprel}")

In [ ]:
# Tourism domain treebank
nlp_tour = Pipeline(
    "tur",
    processors=["tokenize", "pos", "lemma", "depparse"],
    pos_treebank="Tourism",
)
doc = nlp_tour("Otelin konumu mükemmeldi.")
for w in doc.words:
    print(f"{w.text:<20} {w.upos:<8} {w.lemma}")

## 5. Named Entity Recognition (NER)

Turkish NER trained on the Starlang corpus, recognising `PER`, `ORG`, `LOC`, `MISC`.

In [ ]:
nlp_ner = Pipeline("tur", processors=["tokenize", "pos", "lemma", "ner"])

doc = nlp_ner("Ahmet Çelik, Türk Hava Yolları'nda çalışıyor.")
for ent in doc.entities:
    print(f"  {ent.text!r:<30} type={ent.type}")

## 6. Full Pipeline with CoNLL-U Export

In [ ]:
nlp_full = Pipeline(
    "tur",
    processors=["tokenize", "mwt", "pos", "lemma", "depparse", "ner"],
)

doc = nlp_full("Mehmet Yılmaz, Ankara'daki bir şirkette müdür olarak çalışmaktadır.")

# Word-level annotations
for w in doc.words:
    print(
        f"{w.text:<20} upos={w.upos:<8} "
        f"lemma={w.lemma:<15} ner={w.ner:<8} dep={w.deprel}"
    )

# Named entity spans
print("\nEntities:")
for ent in doc.entities:
    print(f"  {ent.text!r} -> {ent.type}")

# CoNLL-U export
print("\nCoNLL-U:")
print(doc.to_conllu())

## 7. Sentence Embeddings and Semantic Similarity (NLLB-200)

In [ ]:
import math

turkicnlp.download("tur", processors=["embeddings"])

embed = Pipeline("tur", processors=["embeddings"])

s1 = "Bugün hava çok güzel ve parkta yürüyüş yaptım."
s2 = "Parkta yürüyüş yapmak bugün çok keyifliydi."
s3 = "Matematik çok zor bir derstir."

d1, d2, d3 = embed(s1), embed(s2), embed(s3)


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x**2 for x in a))
    norm_b = math.sqrt(sum(y**2 for y in b))
    return dot / (norm_a * norm_b)


print(f"sim(s1, s2) = {cosine(d1.embedding, d2.embedding):.4f}  (same topic)")
print(f"sim(s1, s3) = {cosine(d1.embedding, d3.embedding):.4f}  (different topic)")

## 8. Machine Translation (Turkish → English / Kazakh)

In [ ]:
turkicnlp.download("tur", processors=["translate"])

# Turkish -> English
nlp_en = Pipeline(
    "tur", processors=["translate"], translate_tgt_lang="eng_Latn"
)
doc = nlp_en("Türkiye, zengin bir tarihe ve kültüre sahip bir ülkedir.")
print("EN:", doc.translation)

# Turkish -> Kazakh
nlp_kk = Pipeline(
    "tur", processors=["translate"], translate_tgt_lang="kaz_Cyrl"
)
doc = nlp_kk("Merhaba, nasılsınız?")
print("KK:", doc.translation)

## 9. Discover Available Processors

In [ ]:
print("Languages:", turkicnlp.list_languages()[:6], "...")
print("Turkish processors:", turkicnlp.list_processors("tur"))